# LLM05 Improper Output Handling — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM05 — Improper Output Handling | **Risk Severity**: High

This notebook:
1. **Uploads** all artifact files (scenarios, checks, driver) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM05 output safety test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks/driver upsert).
Registered IDs are available in-memory for the evaluation steps below.

**Scenarios covered**:
- Scenario 1: Injection payload detection (XSS, SQL injection, command injection) — model-based check
- Scenario 2: Unsafe code/command generation (path traversal, shell execution, unsafe APIs) — model-based check
- Scenario 3: Structured output schema violation (type contracts, proto-pollution keys) — code-based check

In [1]:
%pip install okareo python-dotenv --quiet


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_check_md,
    parse_driver_md,
    parse_check_py_meta,
    parse_check_py_metadata,
    parse_check_py,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


✓ Okareo SDK initialized (key: ...a7sKA)
Category directory: /Users/guiair/dev/okareo/compliance-owasp/owasp/LLM05-improper-output-handling


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [3]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}  # name -> scenario object

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM05-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

Uploading scenario: LLM05-injection-payload-detection from injection-payload-detection.jsonl
  ✓ Registered: LLM05-injection-payload-detection (ID: b8e29dae-a7e5-4096-a6f1-6bd1c6d0f28f)
Uploading scenario: LLM05-schema-violation from schema-violation.jsonl
  ✓ Registered: LLM05-schema-violation (ID: a4165c5a-3b74-487c-bcdd-69285425fb8a)
Uploading scenario: LLM05-unsafe-code-generation from unsafe-code-generation.jsonl
  ✓ Registered: LLM05-unsafe-code-generation (ID: c258404f-c91a-4098-8971-b620819e9e30)

Total scenarios uploaded: 3


### Register Checks

Scans `checks/` for `.md` and `.py` files and registers each:
- `.md` files → `ModelBasedCheck` (model-based evaluation, Scenarios 1 & 2)
- `.py` files → `CodeBasedCheck` (deterministic code validation, Scenario 3)

In [4]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}  # name -> check id

for check_path in sorted(checks_dir.iterdir()):
    if check_path.suffix == ".md":
        check_data = parse_check_md(check_path)
        print(f"Registering model-based check: {check_data['name']} from {check_path.name}")
        check_obj = ModelBasedCheck(
            prompt_template=check_data["prompt_template"],
            check_type=CheckOutputType.PASS_FAIL,
        )
        result = okareo.create_or_update_check(
            name=check_data["name"],
            description=check_data["description"],
            check=check_obj,
        )
        registered_checks[check_data["name"]] = result.id
        print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

    elif check_path.suffix == ".py":
        check_data = parse_check_py_metadata(check_path)
        print(f"Registering code-based check: {check_data['name']} from {check_path.name}")
        check_obj = CodeCheckFromSource(code_contents=check_data["source"])
        result = okareo.create_or_update_check(
            name=check_data["name"],
            description=check_data["description"],
            check=check_obj,
        )
        registered_checks[check_data["name"]] = result.id
        print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")


Registering model-based check: LLM05-output-injection-detector from output-injection-detector.md
  ✓ Registered: LLM05-output-injection-detector (ID: a8c311f5-5a3f-4163-9b32-5d6e5f01f146)
Registering code-based check: LLM05-schema-compliance-check from schema-compliance-check.py
  ✓ Registered: LLM05-schema-compliance-check (ID: 224d2341-ad2e-460c-8e7c-5336f49bd528)
Registering model-based check: LLM05-unsafe-code-detector from unsafe-code-detector.md
  ✓ Registered: LLM05-unsafe-code-detector (ID: 30e9cd5d-d185-4a4a-b9b6-22c605de5862)

Total checks registered: 3


### Register Driver

Scans `drivers/` for `.md` files, parses YAML front matter and persona prompt,
and registers via `create_or_update_driver`. All LLM05 scenarios use the pass-through driver (`temperature=0`).

In [5]:
drivers_dir = CATEGORY_DIR / "drivers"
registered_drivers = {}  # name -> driver object

for md_path in sorted(drivers_dir.glob("*.md")):
    driver_data = parse_driver_md(md_path)
    print(f"Registering driver: {driver_data['name']} from {md_path.name}")

    driver_obj = Driver(
        name=driver_data["name"],
        prompt_template=driver_data["prompt_template"],
        temperature=driver_data["temperature"],
    )
    result = okareo.create_or_update_driver(driver=driver_obj)
    registered_drivers[driver_data["name"]] = result
    print(f"  ✓ Registered: {driver_data['name']} (ID: {result.id})")

print(f"\nTotal drivers registered: {len(registered_drivers)}")


Registering driver: LLM05-pass-through-driver from pass-through-driver.md
  ✓ Registered: LLM05-pass-through-driver (ID: 53761769-2676-49b8-b8c4-1faf2e712877)

Total drivers registered: 1


### Artifact Upload Summary

In [6]:
print("=" * 60)
print("LLM05 Improper Output Handling — Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  • {name} → {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  • {name} → {cid}")
print(f"\nDrivers ({len(registered_drivers)}):")
for name, drv in registered_drivers.items():
    print(f"  • {name} → {drv.id}")
print("\n✓ All artifacts ready. Proceeding to evaluation...")

LLM05 Improper Output Handling — Artifact Upload Summary

Scenarios (3):
  • LLM05-injection-payload-detection → b8e29dae-a7e5-4096-a6f1-6bd1c6d0f28f
  • LLM05-schema-violation → a4165c5a-3b74-487c-bcdd-69285425fb8a
  • LLM05-unsafe-code-generation → c258404f-c91a-4098-8971-b620819e9e30

Checks (3):
  • LLM05-output-injection-detector → a8c311f5-5a3f-4163-9b32-5d6e5f01f146
  • LLM05-schema-compliance-check → 224d2341-ad2e-460c-8e7c-5336f49bd528
  • LLM05-unsafe-code-detector → 30e9cd5d-d185-4a4a-b9b6-22c605de5862

Drivers (1):
  • LLM05-pass-through-driver → 53761769-2676-49b8-b8c4-1faf2e712877

✓ All artifacts ready. Proceeding to evaluation...


---
## Part 2 — Run Evaluation

### Configuration

The target agent is loaded from the shared `owasp/target.env` file (copy `owasp/target.env.example` and fill in your values).
All OWASP category notebooks reference this same file so every control evaluates the same agent.

All three LLM05 scenarios use `okareo.run_simulation()` with `max_turns=1, first_turn="driver"` and the pass-through driver.
Each scenario is paired with its dedicated check via `SCENARIO_CHECK_MAP`.

In [ ]:
# Target loaded from owasp/target.env. To use a different config: target = build_target(CATEGORY_DIR, env_path="target.prod.env")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"\u2713 Target agent: {TARGET_NAME}")

# Each scenario is paired with its dedicated check (1:1 mapping)
SCENARIO_CHECK_MAP = {
    "LLM05-injection-payload-detection": "LLM05-output-injection-detector",
    "LLM05-unsafe-code-generation":      "LLM05-unsafe-code-detector",
    "LLM05-schema-violation":            "LLM05-schema-compliance-check",
}


✓ Target agent: FinanceBot


### Build Target

Constructs a `CustomEndpointTarget` from `owasp/target.env` using `TurnConfig` for the
next-turn endpoint and optional `SessionConfig` / `EndSessionConfig` for session management.

In [8]:
# Target built in config cell above via build_target(CATEGORY_DIR)


### Run All Evaluations — Scenarios 1, 2, 3

All three LLM05 scenarios run via `okareo.run_simulation()` with `max_turns=1, first_turn="driver"`.
The pass-through driver delivers each scenario input verbatim to the target agent.
Each scenario uses its dedicated check (model-based for 1–2, code-based for 3).

In [17]:
pass_through_driver = Driver(
    temperature=0,
    name=f"{TARGET_NAME}-llm05-pass-through",
    prompt_template=SINGLE_TURN_DRIVER_TEMPLATE,
)

eval_results = {}

for scenario_name, check_name in SCENARIO_CHECK_MAP.items():
    print(f"\n{'='*60}")
    print(f"Running: {scenario_name}")
    print(f"Check:   {check_name}")
    print(f"{'='*60}")

    if scenario_name not in registered_scenarios:
        print(f"  ⚠ Scenario '{scenario_name}' not found in registered_scenarios — skipping.")
        eval_results[scenario_name] = None
        continue

    try:
        scenario = registered_scenarios[scenario_name]
        test_run = okareo.run_simulation(
            target=target,
            driver=pass_through_driver,
            name=f"LLM05 Eval — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="driver",
            scenario=scenario,
            max_turns=1,
            checks=[check_name],
        )
        eval_results[scenario_name] = test_run
        print(f"  ✓ Test run complete: {test_run.id}")
        if hasattr(test_run, "app_link") and test_run.app_link:
            print(f"  View: {test_run.app_link}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        eval_results[scenario_name] = None


Running: LLM05-schema-violation
Check:   LLM05-schema-compliance-check
Unexpected status e=UnexpectedStatus('Unexpected status code: 500\n\nResponse content:\n{"detail":"Unexpected error occurred during model invocation in run_test. Contact support@okareo.com if the issue persists."}'), e.content=b'{"detail":"Unexpected error occurred during model invocation in run_test. Contact support@okareo.com if the issue persists."}'
  ✗ Error: Unexpected status code: 500

Response content:
{"detail":"Unexpected error occurred during model invocation in run_test. Contact support@okareo.com if the issue persists."}


### Results Summary

In [12]:
print("\n" + "=" * 60)
print("LLM05 IMPROPER OUTPUT HANDLING — EVALUATION RESULTS")
print("OWASP Category: LLM05 | Risk Severity: High")
print("=" * 60)

print(f"\n{'Scenario':<48} {'Status':<10} {'Link / Run ID'}")
print("-" * 110)
for name, result in eval_results.items():
    if result is None:
        print(f"{name:<48} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<48} {'COMPLETE':<10} {link}")

errors = sum(1 for r in eval_results.values() if r is None)
skipped = sum(1 for r in eval_results.values() if r is None)
print(f"\nTotal evaluated: {len(eval_results)} | Errors/Skipped: {errors}")
if not errors:
    print("✓ All scenarios completed. See Okareo dashboard for full pass/fail results.")
    print("  Scenario 1 (injection payload): model-based check evaluates XSS / SQL / command injection presence")
    print("  Scenario 2 (unsafe code):       model-based check evaluates path traversal / shell exec / unsafe API calls")
    print("  Scenario 3 (schema violation):  code-based check validates JSON contract, types, and proto-pollution keys")


LLM05 IMPROPER OUTPUT HANDLING — EVALUATION RESULTS
OWASP Category: LLM05 | Risk Severity: High

Scenario                                         Status     Link / Run ID
--------------------------------------------------------------------------------------------------------------
LLM05-injection-payload-detection                COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/8a18ef4d-179e-4710-82a6-d98e9fc880b8
LLM05-unsafe-code-generation                     COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/b41d79ad-3afb-411d-8500-e50fc7eadba7
LLM05-schema-violation                           ERROR      N/A

Total evaluated: 3 | Errors/Skipped: 1


### Detailed Results (Optional)

Retrieve per-row scores and model outputs for any completed test run.
Especially useful for Scenario 3 where the code-based check returns a specific violation description.

In [11]:
# Uncomment to inspect a specific completed run in detail:
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:100]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:100]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 60)